<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/20-llms-in-practice/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Ask the Course

This course has twenty modules, each with a README and a cheatsheet, plus a glossary: about 30,000 words in all. In this project you'll build an assistant that answers questions about the course from those files, with **retrieval-augmented generation**, and, just as importantly, **measure** how well it works.

The plan:

1. Collect the course's README files, cheatsheets and glossary, and split them into chunks.
2. Write a test set: questions, the module that answers each one, and the key fact the answer must contain.
3. Compare four ways of retrieving chunks: TF-IDF, BM25, embeddings, and a hybrid of BM25 and embeddings.
4. Generate answers with SmolLM2, citing their sources, and score them.
5. Study the failures, add a safety net for questions the course doesn't answer, and wrap it all in one `ask` function.

Everything uses free, open models. The language model is **SmolLM2-360M-Instruct**, the bigger sibling of the lesson's model: still small enough for a CPU, but noticeably better at reading. The first run downloads it (about 700 MB). On a CPU the notebook takes several minutes, most of it generating answers; a Colab GPU is faster.

In [ ]:
import importlib.util
import subprocess
import sys

for package, pip_name in [("transformers", "transformers"), ("sentence_transformers", "sentence-transformers")]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import collections
import math
import re
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer

CHAT_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", device)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Categorical colours, always used in this order. The first three are checked to stay
# distinguishable for the common forms of colour blindness, even in scatter plots.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]

SURFACE = "#fcfcfb"     # chart background
INK = "#0b0b0b"         # titles and important text
INK_2 = "#52514e"       # labels and tick text
MUTED = "#898781"       # de-emphasised marks
GRID = "#e1e0d9"        # gridlines
AXIS = "#c3c2b7"        # axis lines

# Sequential: one hue, light to dark, for magnitude (heatmaps, counts)
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "course_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
# Diverging: two opposite hues with a neutral grey in the middle, for values around zero
DIVERGING = LinearSegmentedColormap.from_list(
    "course_diverging", ["#184f95", "#86b6ef", "#f0efec", "#f0a3a2", "#c23a3a"])


def use_course_style():
    for cmap in (SEQUENTIAL, DIVERGING):
        if cmap.name not in mpl.colormaps:
            mpl.colormaps.register(cmap)
    mpl.rcParams.update({
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "figure.dpi": 100,
        "savefig.dpi": 150,
        "figure.figsize": (7, 4),
        "font.size": 10,
        "text.color": INK,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.labelcolor": INK_2,
        "axes.edgecolor": AXIS,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "grid.linestyle": "-",
        "xtick.color": AXIS,
        "ytick.color": AXIS,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "lines.linewidth": 1.6,
        "lines.solid_capstyle": "round",
        "lines.markersize": 6,
        "patch.linewidth": 0,
        "legend.frameon": False,
        "legend.labelcolor": INK_2,
        "image.cmap": "course_blues",
    })


use_course_style()

## 1. The documents

If you're running this inside a copy of the course on your computer, the files are read from the neighbouring folders; otherwise (on Colab, for example) they're downloaded from the course's GitHub repository. A file that can't be found is skipped and reported.

In [ ]:
REPO = "https://raw.githubusercontent.com/anupbagale/python-for-ai-ml/main/"
FOLDERS = ["00-start-here", "01-python-foundations", "02-data-structures", "03-oop-for-ml", "04-pythonic-tools", "05-files-and-data",
           "06-numpy", "07-pandas", "08-visualization", "09-math-in-code", "10-engineering-practices", "11-capstone",
           "12-supervised-learning", "13-evaluating-classifiers", "14-trees-and-ensembles", "15-unsupervised-learning",
           "16-time-series-and-recommendations", "17-deep-learning-pytorch", "18-computer-vision",
           "19-sequences-and-transformers", "20-llms-in-practice"]


def read_course_file(folder, name):
    local = Path("..") / folder / name
    if local.exists():
        return local.read_text(encoding="utf-8")
    try:
        return urllib.request.urlopen(REPO + f"{folder}/{name}", timeout=30).read().decode("utf-8")
    except Exception:
        return None


documents, missing = [], []
for folder in FOLDERS:
    for name in ["README.md", "cheatsheet.md"] + (["glossary.md"] if folder == "00-start-here" else []):
        text = read_course_file(folder, name)
        if text is None:
            if name != "cheatsheet.md" or folder != "00-start-here":
                missing.append(f"{folder}/{name}")
            continue
        documents.append({"folder": folder, "file": name, "text": text})

print(f"{len(documents)} files, {sum(len(d['text'].split()) for d in documents):,} words")
if missing:
    print("not found:", missing)

Each file is split at its `## ` headings, so each chunk is one section about one topic. Sections longer than 200 words are cut further into overlapping pieces (Exercise 3's `chunk_words`). Every chunk starts with its module and section title, which gives both the search and the language model useful context: "Mini project" means little on its own, but "17-deep-learning-pytorch, README: Mini project" says exactly what it is.

In [ ]:
def chunk_words(text, size, overlap):
    words = text.split()
    chunks = []
    for start in range(0, len(words), size - overlap):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks


chunks = []
for doc in documents:
    sections = re.split(r"\n(?=## )", doc["text"])
    for section in sections:
        lines = section.strip().split("\n", 1)
        heading = lines[0].lstrip("# ").strip()
        body = lines[1].strip() if len(lines) > 1 else ""
        if not body:
            continue
        for piece in chunk_words(body, size=200, overlap=40):
            chunks.append({"folder": doc["folder"], "file": doc["file"], "heading": heading,
                           "text": f"{doc['folder']}, {doc['file']}: {heading}. {piece}"})

lengths = [len(c["text"].split()) for c in chunks]
print(f"{len(chunks)} chunks, {np.mean(lengths):.0f} words on average, longest {max(lengths)}")
print("\nexample:\n", chunks[len(chunks) // 2]["text"][:400])

## 2. A test set

Before building anything, write down what "working" means. Each test question below comes with the folder that answers it (sometimes two) and the key facts a correct answer should contain. The questions use everyday wording on purpose, not the README's exact phrases, because that's how people ask.

Writing these is the slowest part of the project, and the most useful: every decision that follows (which search method, how many chunks, which prompt) will be made by measuring against them.

In [ ]:
TESTS = [
    ("Which algorithm does the first module's fruit project use?", {"01-python-foundations"}, ["nearest", "k-nn"]),
    ("How does the spam filter avoid multiplying lots of tiny probabilities?", {"02-data-structures"}, ["log"]),
    ("Which class records calculations so it can compute gradients in the OOP module?", {"03-oop-for-ml"}, ["value"]),
    ("Which tool creates the hyperparameter search space lazily?", {"04-pythonic-tools"}, ["product"]),
    ("What is SHA-256 used for in the files and data project?", {"05-files-and-data"}, ["checksum", "training data"]),
    ("Which dataset does the NumPy neural network learn from?", {"06-numpy"}, ["digit"]),
    ("What kind of business does the pandas project simulate?", {"07-pandas"}, ["shop", "customer", "order"]),
    ("How many tests are in the engineering practices project?", {"10-engineering-practices"}, ["28"]),
    ("How many steps does Newton's method need in the maths module?", {"09-math-in-code"}, ["8", "eight"]),
    ("What does the capstone web service predict?", {"11-capstone"}, ["sex", "penguin"]),
    ("Which houses are predicted in the supervised learning project?", {"12-supervised-learning"}, ["california"]),
    ("Why was the length of the phone call removed in the bank project?", {"13-evaluating-classifiers"}, ["leak", "after the call"]),
    ("How much does a random forest reduce the error compared with the linear models?", {"14-trees-and-ensembles"}, ["15"]),
    ("How many wholesale customers are in the clustering project?", {"15-unsupervised-learning"}, ["440"]),
    ("What is forecasted hour by hour in Module 16?", {"16-time-series-and-recommendations"}, ["bike"]),
    ("Which clothing dataset is used in the PyTorch module?", {"17-deep-learning-pytorch"}, ["fashion"]),
    ("What does the recycling sorter look at to decide?", {"18-computer-vision"}, ["photo", "image", "picture"]),
    ("Why does writing the answer backwards help a transformer add?", {"19-sequences-and-transformers"}, ["carry", "units", "least significant"]),
    ("Which text is used to train the small GPT?", {"19-sequences-and-transformers"}, ["shakespeare"]),
    ("What does LoRA stand for?", {"20-llms-in-practice", "00-start-here"}, ["low-rank", "low rank"]),
    ("What does perplexity measure?", {"00-start-here", "19-sequences-and-transformers"}, ["cross-entropy", "unsure", "choosing"]),
    ("What is overfitting?", {"00-start-here"}, ["new data", "unseen", "memoris"]),
    ("Which learning-rate schedule rises and then decays?", {"17-deep-learning-pytorch"}, ["one-cycle", "onecycle", "warmup"]),
    ("Which dataset of rubbish photos does the computer vision project use?", {"18-computer-vision"}, ["trashnet"]),
]
print(f"{len(TESTS)} test questions")

## 3. Four ways to search

All four methods give every chunk a score for a question; the top chunks are what the model will read.

- **TF-IDF**: word overlap, with rare words counting more (scikit-learn).
- **BM25**: the classic search-engine formula from Exercise 10, which also limits the effect of repeated words and long chunks.
- **Embeddings**: cosine similarity between `all-MiniLM-L6-v2` vectors, matching by meaning.
- **Hybrid**: combine the BM25 and embedding rankings with **reciprocal rank fusion**: each chunk scores `1 / (60 + rank)` in each list, and the two are added. It uses only the ranks, so it doesn't matter that the two methods' scores are on completely different scales.

In [ ]:
texts = [c["text"] for c in chunks]


def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())


# TF-IDF
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit(texts)
tfidf_matrix = tfidf.transform(texts)

# BM25
chunk_words_list = [words(t) for t in texts]
chunk_counts = [collections.Counter(w) for w in chunk_words_list]
chunk_lengths = np.array([len(w) for w in chunk_words_list], dtype=float)
doc_freq = collections.Counter(word for c in chunk_counts for word in c)


def bm25(query, k1=1.5, b=0.75):
    scores = np.zeros(len(texts))
    for q in words(query):
        idf = math.log((len(texts) - doc_freq[q] + 0.5) / (doc_freq[q] + 0.5) + 1)
        f = np.array([c[q] for c in chunk_counts], dtype=float)
        scores += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * chunk_lengths / chunk_lengths.mean()))
    return scores


# Embeddings
start = time.time()
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)
chunk_vectors = embedder.encode(texts, normalize_embeddings=True, batch_size=64)
print(f"embedded {len(texts)} chunks in {time.time() - start:.0f} seconds")


def rank(scores):
    return list(np.argsort(-scores, kind="stable"))


def search(question, method, k=3):
    if method == "TF-IDF":
        scores = (tfidf.transform([question]) @ tfidf_matrix.T).toarray()[0]
    elif method == "BM25":
        scores = bm25(question)
    elif method == "embeddings":
        scores = chunk_vectors @ embedder.encode([question], normalize_embeddings=True)[0]
    elif method == "hybrid":
        scores = np.zeros(len(texts))
        for ranking in [rank(bm25(question)), rank(chunk_vectors @ embedder.encode([question], normalize_embeddings=True)[0])]:
            for position, i in enumerate(ranking):
                scores[i] += 1 / (60 + position + 1)
    return rank(scores)[:k], scores

Now the evaluation from Exercise 6. A question counts as found if one of the top `k` chunks comes from a folder that answers it; MRR rewards finding it near the top.

In [ ]:
def evaluate_retrieval(method, k=3):
    hits, reciprocal_ranks, failures = 0, [], []
    for question, folders, _ in TESTS:
        top, scores = search(question, method, k=len(texts))
        ranked_folders = [chunks[i]["folder"] for i in top]
        hits += any(f in folders for f in ranked_folders[:k])
        first = next((pos for pos, f in enumerate(ranked_folders, start=1) if f in folders), None)
        reciprocal_ranks.append(1 / first if first else 0.0)
        if not any(f in folders for f in ranked_folders[:k]):
            failures.append((question, ranked_folders[:k]))
    return hits / len(TESTS), float(np.mean(reciprocal_ranks)), failures


METHODS = ["TF-IDF", "BM25", "embeddings", "hybrid"]
retrieval = {m: evaluate_retrieval(m) for m in METHODS}
print(f"{'method':<12}{'recall@3':>10}{'MRR':>8}")
for m in METHODS:
    print(f"{m:<12}{retrieval[m][0]:>10.2f}{retrieval[m][1]:>8.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(len(METHODS))
ax.bar(x - 0.18, [retrieval[m][0] for m in METHODS], width=0.34, color=PALETTE[0], label="recall@3")
ax.bar(x + 0.18, [retrieval[m][1] for m in METHODS], width=0.34, color=PALETTE[1], label="MRR")
ax.set_xticks(x, METHODS)
ax.set_ylim(0, 1)
ax.set_ylabel("score on the test questions")
ax.set_title("Retrieval quality of four search methods")
ax.legend(loc="lower right")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

Read the table before trusting it: with 24 questions, one question is about 4 percentage points, so small differences are noise. Look instead for clear gaps, and for the **kind** of question each method misses:

In [ ]:
best = max(METHODS, key=lambda m: (retrieval[m][0], retrieval[m][1]))
print(f"questions the {best} search missed (top 3 folders shown):")
for question, top_folders in retrieval[best][2]:
    print(f"  {question}\n      -> {top_folders}")
for m in METHODS:
    print(f"\n{m} missed {len(retrieval[m][2])}: {[q[:45] for q, _ in retrieval[m][2]]}")

Typically, keyword methods miss questions phrased in different words from the files, while embeddings can miss questions that hinge on a specific name, number or rare word, which is exactly what keyword methods are good at. Combining the two is often the most reliable choice, which is why hybrid search is so common in practice. Whatever your table shows, the method to use is the one that does best **on your own test set**, and the failures tell you what to try next: better chunking, more chunks, or rewording the stored text.

## 4. Generating answers

Now the language model. It gets the top 3 chunks from the hybrid search and an instruction to answer from them only. Two things help small models a lot here. The chunks are **cleaned** before the model sees them: markdown symbols, links and code blocks are useful to people reading the files but are just noise in a prompt. And each chunk is labelled with a readable name ("Module 17, Mini project") instead of a folder name. The program, not the model, keeps track of which chunks were used, so the sources shown to the reader are always accurate.

Which search method to use is your call, based on your table. The code below uses the hybrid search; change `METHOD` if another did clearly better.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL).to(device).eval()

SYSTEM = ("You answer questions about a Python and machine learning course. Use only the notes given. "
          "Answer in one short sentence. If the notes don't contain the answer, say you don't know.")
METHOD = "hybrid"


def clean(text):
    """Remove markdown that helps people but distracts a small model: links, code blocks, symbols."""
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)          # [text](link) -> text
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)              # code blocks
    text = re.sub(r"[`*|#>]", " ", text)
    text = re.sub(r"-{3,}", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def label(chunk):
    module = "Start Here" if chunk["folder"].startswith("00") else f"Module {int(chunk['folder'][:2])}"
    kind = {"README.md": "overview", "cheatsheet.md": "cheatsheet", "glossary.md": "glossary"}[chunk["file"]]
    return f"{module} {kind}, {chunk['heading']}"


def generate(messages, max_new_tokens=80):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def rag_answer(question, method=METHOD, k=3):
    top, scores = search(question, method, k=k)
    notes = "\n\n".join(f"Note {n} ({label(chunks[i])}): {clean(chunks[i]['text'].split(': ', 1)[1])}"
                         for n, i in enumerate(top, start=1))
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"{notes}\n\nQuestion: {question}"}]
    return generate(messages), top


for question, _, _ in TESTS[:4]:
    reply, top = rag_answer(question)
    print(f"Q: {question}\nA: {reply}")
    print("   sources:", [label(chunks[i]) for i in top], "\n")

Now all 24 questions, scored by checking for the key facts (crude, but automatic). For comparison, the same questions without any sources show what the model "knows" on its own about this course, which should be very little.

In [ ]:
def contains_fact(reply, facts):
    return any(fact in reply.lower() for fact in facts)


start = time.time()
rows = []
for question, folders, facts in TESTS:
    reply, top = rag_answer(question)
    retrieved = any(chunks[i]["folder"] in folders for i in top)
    closed_book = generate([{"role": "user", "content": question}], max_new_tokens=60)
    rows.append((question, retrieved, contains_fact(reply, facts), contains_fact(closed_book, facts), reply))
print(f"({time.time() - start:.0f} seconds)\n")

for question, retrieved, correct, closed, reply in rows:
    print(f"{'R' if retrieved else '.'} {'A' if correct else '.'}  {question[:58]:<58} {reply[:60]!r}")

n = len(rows)
print(f"\nright source retrieved:           {sum(r[1] for r in rows) / n:.0%}")
print(f"answer contains the key fact:     {sum(r[2] for r in rows) / n:.0%}")
print(f"without sources (closed book):    {sum(r[3] for r in rows) / n:.0%}")
print(f"right source, but answer missed:  {sum(r[1] and not r[2] for r in rows)}")

The columns separate the two ways the system can fail. `R` means the right folder was among the retrieved chunks; `A` means the answer contains a key fact.

- **R without A**: the information was in front of the model and it didn't use it. That's a reading failure, the main weakness of small models. A larger model is the most direct fix; a shorter, more focused prompt (fewer, better chunks) can also help.
- **Neither**: retrieval failed first, and no model can answer from the wrong text.
- **A with R**: read a few of these too. The keyword check can be fooled, and a right fact can come with a wrong claim around it.

The closed-book column shows why retrieval matters: the model knows nothing about this particular course, so almost everything it gets right, it gets from the sources.

## 5. Knowing when not to answer

Some questions aren't answered anywhere in the course. A good assistant says so instead of improvising. One simple safety net uses the embedding similarity of the best chunk: if even the best match is weak, don't ask the model at all. To choose the threshold, compare the best similarity for questions the course answers with questions it doesn't:

In [ ]:
OFF_TOPIC = ["What is the capital of Australia?", "How do I bake sourdough bread?", "Who won the 2010 football World Cup?",
             "What is the best phone to buy?", "How tall is Mount Everest?", "Can you recommend a good holiday destination?"]


def best_similarity(question):
    return float((chunk_vectors @ embedder.encode([question], normalize_embeddings=True)[0]).max())


on_topic_scores = [best_similarity(q) for q, _, _ in TESTS]
off_topic_scores = [best_similarity(q) for q in OFF_TOPIC]
print(f"course questions: lowest best match {min(on_topic_scores):.2f}, median {np.median(on_topic_scores):.2f}")
print(f"off-topic:        highest best match {max(off_topic_scores):.2f}, median {np.median(off_topic_scores):.2f}")
THRESHOLD = (min(on_topic_scores) + max(off_topic_scores)) / 2
print(f"threshold halfway between: {THRESHOLD:.2f}")

If the two groups don't overlap, a threshold between them separates them on these examples. If they do overlap, no threshold is perfect, and you have to choose which mistake you'd rather make. Either way, with only 30 questions, treat it as a starting point: real questions will land near the threshold, so collect them and adjust. Whether it's worse to refuse a fair question or to answer an off-topic one is a decision for the people the assistant serves, and it moves the threshold up or down.

## 6. The assistant

Putting it together: search, the safety net, generation, and the sources shown with every answer so the reader can check.

In [ ]:
def ask(question, k=3):
    if best_similarity(question) < THRESHOLD:
        return "I couldn't find anything about that in the course materials."
    reply, top = rag_answer(question, k=k)
    sources = [label(chunks[i]) for i in top]
    return reply + "\n   sources: " + "; ".join(sources)


for question in ["Which module teaches convolutional networks?", "Where can I learn about cross-validation?",
                 "How do I make a good cup of coffee?"]:
    print(f"Q: {question}\nA: {ask(question)}\n")

Try your own questions with `ask("...")`. You'll find that it does best with specific questions whose answer sits in one section, and struggles with questions that need information combined from many places ("Which modules use the penguin data?"), which is a known weakness of simple RAG.

## What we learned

- **A test set comes first.** Two dozen questions with known answers turned every choice (search method, safety threshold) into a measurement instead of a guess.
- **Measure retrieval and generation separately.** They fail for different reasons and need different fixes.
- **Keyword and embedding search fail on different questions**; hybrid search combines their strengths.
- **The model only knows what it's given.** Closed-book answers about this course are almost always wrong; with the right sources, the model can find the fact, and bigger models find it far more reliably.
- **Show sources** so people can check answers, and **refuse** when retrieval finds nothing relevant.

**Try next:**

- Change the chunk size (100, 200, 400 words) and the number of retrieved chunks, and watch recall@3 and the answer score.
- Swap the language model: `HuggingFaceTB/SmolLM2-135M-Instruct` (smaller, faster) or `HuggingFaceTB/SmolLM2-1.7B-Instruct` (bigger, best on a GPU). Compare the "R without A" count.
- Try sending the raw chunks without `clean` and see whether the answers change.
- Add ten questions of your own, including some that need two modules, and see which part of the system fails.
- Use the `answer_f1` from Exercise 7 instead of the key-fact check, with a short reference answer for each question.

Module 21 closes the course with the other half of building AI responsibly: fairness, interpretability, privacy, and running models reliably in production.